# Unified image & video generation with Bernini-R-1.3B and OpenVINO

[Bernini-R-1.3B](https://huggingface.co/ByteDance/Bernini-R-1.3B-Diffusers) is a unified, multi-task diffusion *renderer* from ByteDance. A single model performs **text-to-image (t2i)**, **image editing (i2i)**, **text-to-video (t2v)**, **video editing (v2v)** and **reference-image-to-video (r2v / rv2v)** generation. It is fine-tuned from [Wan2.1-1.3B](https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B) and re-uses the Wan components: a `WanTransformer3DModel` diffusion transformer, a `UMT5EncoderModel` text encoder, an `AutoencoderKLWan` spatio-temporal VAE and a `UniPCMultistepScheduler`.

In this notebook we convert, optimize and run Bernini-R with OpenVINO. All of Bernini's data-dependent generation logic (the per-step denoising loop, the seven guidance modes, the source-id rotary embeddings, the per-step token assembly) is kept in python; only the heavy leaf compute -- the text encoder, the transformer block stack (`BlocksCore`), and the VAE encoder/decoder -- runs as OpenVINO static graphs. See [README.md](README.md) for the full split rationale.

#### Table of contents:
- [Prerequisites](#Prerequisites)
- [Download the model](#Download-the-model)
- [Convert and optimize the model](#Convert-and-optimize-the-model)
- [Select inference device](#Select-inference-device)
- [Build the OpenVINO pipeline](#Build-the-OpenVINO-pipeline)
- [Run the tasks](#Run-the-tasks) (t2i / t2v / i2i / v2v / r2v / rv2v)
- [Interactive demo](#Interactive-demo)


## Prerequisites
[back to top](#Table-of-contents:)

Bernini's reference code pins `diffusers==0.35.2` and `transformers==4.57.3`. The cell below installs them together with the `bernini` package and OpenVINO. **If your environment already had different versions of `diffusers`/`transformers`, restart the kernel after this cell before continuing.**

In [ ]:
import platform

%pip install -q "torch>=2.4.0" "torchvision" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "diffusers==0.35.2" "transformers==4.57.3" "accelerate==0.34.2" "safetensors" "einops" "ftfy" "sentencepiece" "imageio" "imageio-ffmpeg" "decord" "tqdm"
%pip install -q "openvino>=2025.0.0" "nncf>=2.15.0" "gradio>=4.19" "huggingface_hub" "modelscope"
%pip install -q "git+https://github.com/bytedance/Bernini.git" --no-deps

if platform.system() == "Windows":
    %pip install -q "truststore"  # use the system trust store behind corporate proxies


In [ ]:
import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

from notebook_utils import device_widget, quantization_widget, collect_telemetry

collect_telemetry("bernini-r-image-video.ipynb")


## Download the model
[back to top](#Table-of-contents:)

Download the [Bernini-R-1.3B-Diffusers](https://huggingface.co/ByteDance/Bernini-R-1.3B-Diffusers) snapshot (~12 GB). The default uses the Hugging Face Hub; set `USE_MODELSCOPE = True` if the Hub is not reachable from your network.

In [ ]:
from pathlib import Path

MODEL_DIR = Path("Bernini-R-1.3B-Diffusers")
USE_MODELSCOPE = False  # set True to download from modelscope.cn instead of huggingface.co

if not (MODEL_DIR / "config.json").exists():
    if USE_MODELSCOPE:
        from modelscope import snapshot_download
        snapshot_download("bytedance-community/Bernini-R-1.3B-Diffusers", local_dir=MODEL_DIR.as_posix())
    else:
        from huggingface_hub import snapshot_download
        snapshot_download("ByteDance/Bernini-R-1.3B-Diffusers", local_dir=MODEL_DIR.as_posix())
print("model in", MODEL_DIR.resolve())


## Convert and optimize the model
[back to top](#Table-of-contents:)

Choose the weight-compression format. The `UMT5` text encoder alone is ~5 GB in FP16, so weight compression noticeably reduces the footprint. **INT8** is a good default; **INT4** is smaller but may drift slightly from the reference; **FP16** is closest to the original and the largest on disk.

In [ ]:
import ipywidgets as widgets

model_format = widgets.Dropdown(
    options=["FP16", "INT8", "INT4"],
    value="INT8",
    description="Weights:",
)
model_format


In [ ]:
import nncf

if model_format.value == "INT4":
    compression_config = {"mode": nncf.CompressWeightsMode.INT4_ASYM, "group_size": 64, "ratio": 1.0}
elif model_format.value == "INT8":
    compression_config = {"mode": nncf.CompressWeightsMode.INT8_ASYM}
else:
    compression_config = None

OV_DIR = Path(f"bernini_ov_{model_format.value.lower()}")
print("OpenVINO IR will be written to", OV_DIR)


The conversion writes one graph for the text encoder, one for the transformer block stack (`BlocksCore`, dynamic token length), and one VAE encoder/decoder graph per temporal latent length. `vae_latent_frames=(1,)` covers images and single frames; add more lengths (e.g. `13` for ~49 video frames) to pre-build video VAE graphs -- they are otherwise compiled lazily on first use.

In [ ]:
from ov_bernini_helper import convert_pipeline

convert_pipeline(MODEL_DIR, OV_DIR, compression_config=compression_config, vae_latent_frames=(1, 13))


## Select inference device
[back to top](#Table-of-contents:)

You can place each component on a different device. The transformer is the compute-heavy part (run once per guidance forward, several times per step).

In [ ]:
device_transformer = device_widget(default="AUTO", description="Transformer:")
device_text_encoder = device_widget(default="AUTO", description="Text encoder:")
device_vae = device_widget(default="AUTO", description="VAE:")
widgets.VBox([device_transformer, device_text_encoder, device_vae])


## Build the OpenVINO pipeline
[back to top](#Table-of-contents:)

In [ ]:
from ov_bernini_helper import load_ov_pipeline

device_map = {
    "transformer": device_transformer.value,
    "text_encoder": device_text_encoder.value,
    "vae": device_vae.value,
}
ov_pipe = load_ov_pipeline(MODEL_DIR, OV_DIR, device_map=device_map, compression_config=compression_config)


## Run the tasks
[back to top](#Table-of-contents:)

Bernini-R is multi-task. Each task uses a specific guidance mode and system prompt; these are pre-defined in `ov_bernini_helper.TASK_GUIDANCE` / `TASK_SYSTEM_PROMPT` (matching the reference `bytedance/Bernini` testcases) and exposed through this small helper. The pipeline writes the result to `output_path` and returns the path -- a PNG for a single frame, an mp4 otherwise.

| task | what it does | extra input |
|------|--------------|-------------|
| `t2i` | text -> image | - |
| `t2v` | text -> video | - |
| `i2i` | image editing | `image` |
| `v2v` | video editing | `video` |
| `r2v` | reference image(s) -> video | `images` |
| `rv2v` | reference image(s) + source video -> video | `video`, `images` |

In [ ]:
from ov_bernini_helper import TASK_GUIDANCE, TASK_SYSTEM_PROMPT

def generate(task, prompt, num_frames=1, steps=40, seed=42, output_path=None, **inputs):
    ext = "png" if (task in ("t2i", "i2i") or num_frames == 1) else "mp4"
    return ov_pipe(
        prompt=prompt,
        guidance_mode=TASK_GUIDANCE[task],
        system_prompt=TASK_SYSTEM_PROMPT[task],
        num_frames=num_frames,
        height=480,
        width=832,
        num_inference_steps=steps,
        seed=seed,
        output_path=output_path or f"{task}.{ext}",
        **inputs,
    )


### Text-to-image

In [ ]:
from PIL import Image

out_path = generate(
    "t2i",
    "Astronaut in a jungle, cold color palette, muted colors, detailed, 8k",
    num_frames=1,
)
Image.open(out_path)


### Text-to-video
Uses the same transformer/VAE graphs with `num_frames > 1`. The first call at a new temporal length compiles the matching VAE graph if it was not pre-built above.

In [ ]:
from IPython.display import Video

video_path = generate("t2v", "A cat walks on the grass, realistic", num_frames=49, seed=42)
Video(video_path, embed=True)


### Image editing (i2i)
Edit a source image with an instruction prompt. The VAE encoder embeds the source image, and the diffusion sampler renders the edited result.

In [ ]:
from notebook_utils import download_file

src = download_file(
    "https://raw.githubusercontent.com/bytedance/Bernini/main/assets/testcases/i2i/source.png",
    "i2i_source.png",
)
edited = generate(
    "i2i",
    "Change the season to a snowy winter scene, keeping the composition unchanged.",
    num_frames=1,
    image=str(src),
)
Image.open(edited)


### Video editing (v2v)
Edit a source video with an instruction prompt (e.g. add or modify an object) while preserving the rest of the scene.

In [ ]:
src_v = download_file(
    "https://raw.githubusercontent.com/bytedance/Bernini/main/assets/testcases/v2v/source_case1.mp4",
    "v2v_source.mp4",
)
edited_v = generate(
    "v2v",
    "Add a realistic snowman beside the path, matching the lighting and snow.",
    num_frames=49,
    steps=30,
    video=str(src_v),
)
Video(edited_v, embed=True)


> Reference-to-video (`r2v`) and reference+video editing (`rv2v`) work the same way -- pass `images=[...]` (and `video=...` for `rv2v`). They are available in the interactive demo below.

## Interactive demo
[back to top](#Table-of-contents:)

The Gradio demo exposes all six tasks. Image / video conditioning inputs appear for the tasks that use them.

In [ ]:
from gradio_helper import make_demo

demo = make_demo(ov_pipe)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)
